# Supplementary Experiment: Improving Synthetic Data Quality

## Purpose

This notebook presents an additional experiment to improve the quality of synthetic Pima diabetes profiles for COMP8240 Assignment 3. It supplements the original synthetic-data experiment rather than replacing it.

In the original experiment, features were generated independently, which may not preserve the relationships between real clinical measurements. In this supplementary experiment, complete feature combinations are sampled from real Pima records and small numerical variations are introduced. This approach aims to preserve more realistic relationships between features.

The experiment generates 200 synthetic profiles and evaluates their quality using missing-value checks, duplicate checks, and comparisons of feature correlations with the original data.

## Dataset Selection and Limitations

Only donor records with nonzero values for glucose, blood pressure, skin thickness, insulin, and BMI are selected. This reduces the use of records containing zero values that may represent missing clinical measurements.

However, this selection changes the characteristics of the donor population and may introduce selection bias. The resulting synthetic profiles may therefore not fully represent the original Pima dataset.

Furthermore, the generated profiles do not have independently verified diabetes outcomes. They are intended for supplementary data-quality analysis and model prediction, not for measuring clinical diagnostic accuracy.

## Relationship to the Main Assignment

The original synthetic experiment and its reported results remain unchanged. This supplementary experiment provides additional evidence about synthetic-data quality, particularly the preservation of relationships between clinical features.




In [5]:
from IPython.display import display
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp

SEED = 42
rng = np.random.default_rng(SEED)
raw = pd.read_csv('diabetes.csv')  # In Colab: upload diabetes.csv first
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age']
assert all(c in raw.columns for c in features + ['Outcome'])
train, holdout = train_test_split(raw, test_size=0.2, random_state=42, stratify=raw['Outcome'])
print('Training rows:', len(train), 'Unused holdout:', len(holdout))

Training rows: 614 Unused holdout: 154


In [6]:
# Build profiles from entire observed training rows, preserving feature relationships.
# Use only rows with physiologically valid nonzero values in these fields.
medical = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
valid = train.loc[(train[medical] > 0).all(axis=1), features].copy()
assert len(valid) >= 30, 'Not enough complete valid rows to construct the cohort'

n = 200
indices = rng.integers(0, len(valid), size=n)
synth = valid.iloc[indices].reset_index(drop=True).copy()
# Small bounded jitter creates distinct hypothetical profiles while retaining joint structure.
# These scales are illustrative and not clinically validated measurement-error models.
scales = {'Glucose':3.0,'BloodPressure':2.0,'SkinThickness':1.0,'Insulin':5.0,'BMI':0.5,'DiabetesPedigreeFunction':0.02}
for col, sd in scales.items():
    synth[col] = synth[col].to_numpy() + rng.normal(0, sd, n)
for col in ['Glucose','BloodPressure','SkinThickness','Insulin']:
    synth[col] = np.maximum(1, np.rint(synth[col])).astype(int)
synth['BMI'] = np.maximum(1, synth['BMI']).round(1)
synth['DiabetesPedigreeFunction'] = np.maximum(0.001, synth['DiabetesPedigreeFunction']).round(3)
for col in ['Pregnancies','Age']:
    synth[col] = synth[col].astype(int)

assert synth.shape == (200,8)
assert not synth.isna().any().any()
assert (synth[medical] > 0).all().all()
print('Synthetic rows:',len(synth),'exact duplicates:',int(synth.duplicated().sum()))
display(synth.head())
synth.to_csv('COMP8240_correlated_synthetic_profiles_200.csv',index=False)

Synthetic rows: 200 exact duplicates: 0


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,5,124,70,38,72,34.3,0.288,28
1,2,122,61,19,139,33.0,0.113,31
2,1,86,81,28,38,34.8,0.124,22
3,1,101,63,16,139,23.3,0.499,22
4,5,98,56,27,78,33.9,0.511,30


In [7]:
# Transparent audit: compare with the valid training donor population (not test data).
# KS values closer to zero indicate closer *marginal* distributions; they are not quality guarantees.
summary = []
for col in features:
    ks = ks_2samp(valid[col], synth[col]).statistic
    summary.append({'feature':col,'train_mean':round(valid[col].mean(),3),
                    'synthetic_mean':round(synth[col].mean(),3),
                    'train_std':round(valid[col].std(),3),
                    'synthetic_std':round(synth[col].std(),3),
                    'KS_statistic':round(float(ks),3)})
quality = pd.DataFrame(summary)
display(quality)
cor_train = valid.corr(numeric_only=True)
cor_synth = synth.corr(numeric_only=True)
mask = np.triu(np.ones(cor_train.shape, dtype=bool),1)
mean_abs_corr_gap = np.abs(cor_train.to_numpy()-cor_synth.to_numpy())[mask].mean()
print('Mean absolute pairwise-correlation gap:',round(float(mean_abs_corr_gap),3))
print('Synthetic exact duplicates:',int(synth.duplicated().sum()))
print('Exact synthetic rows also in donor data:',int(synth.merge(valid.drop_duplicates(),on=features,how='inner').shape[0]))
quality.to_csv('COMP8240_synthetic_quality_audit.csv',index=False)

,feature,train_mean,synthetic_mean,train_std,synthetic_std,KS_statistic
0,Pregnancies,3.261,3.065,3.105,2.942,0.035
1,Glucose,122.758,120.965,30.849,30.893,0.084
2,BloodPressure,70.317,69.300,12.617,12.427,0.067
3,SkinThickness,29.084,29.315,10.439,10.793,0.031
4,Insulin,149.655,149.515,107.284,106.738,0.033
5,BMI,32.911,33.256,7.031,7.448,0.038
6,DiabetesPedigreeFunction,0.528,0.519,0.337,0.329,0.050
7,Age,31.099,30.100,10.262,9.101,0.058


Mean absolute pairwise-correlation gap: 0.045
Synthetic exact duplicates: 0
Exact synthetic rows also in donor data: 0
